In [1]:
# Necessary libraries and useful parameters
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.colorbar as cb

import scipy.sparse as sp
import warnings
warnings.simplefilter("ignore", RuntimeWarning)

newparams = {'font.family': 'cmr10', 'mathtext.fontset': 'cm',
             'axes.grid': False, 'axes.labelsize': 22,
             'xtick.labelsize': 18, 'ytick.labelsize': 18,
             'legend.fontsize': 18, 'axes.titlesize': 20,
             'figure.figsize': (9,6), 'lines.linewidth': 2.5,
             'axes.formatter.use_mathtext': True}
plt.rcParams.update(newparams)
%matplotlib inline

In [2]:
def save_figure(name, folder='figures'):
    import os
    if not os.path.exists(folder):
        os.makedirs(folder)
    plt.savefig(f'{folder}/{name}.pdf', format='pdf', bbox_inches='tight')
    plt.close()

In [3]:
# Color maps
concentration_colors = ['#00274D', '#1B4F72', '#CFA6A6', '#CD5C5C', '#8B3A3A', '#641010']
concentration_cmap = mcolors.LinearSegmentedColormap.from_list('custom_concentrations', concentration_colors)

deposition_colors = ['#F6CCCC', '#CD5C5C', '#641010']
deposition_cmap = mcolors.LinearSegmentedColormap.from_list('custom_reds', deposition_colors)

### Ebola disease chart

In [4]:
# Sample data from CDC.
years = np.array(['76', '77', '79', '89', '94', '95', '96', '00', '01', '03', '04', '05', '07', '08', '11', '12', '14', '17', '18', '20', '21', '22'])
cases = np.array([603, 1, 34, 7, 52, 315, 94, 425, 124, 178, 18, 12, 395, 38, 1, 55, 28715, 8, 3524, 130, 46, 170])
deaths = np.array([431, 1, 22, 0, 31, 254, 68, 224, 97, 157, 18, 10, 229, 15, 1, 20, 11372, 4, 2320, 55, 27, 61])

x = np.arange(len(years))

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - 0.2, deaths, width=0.4, label='Deaths', color='#641010', edgecolor='black')
ax.bar(x + 0.2, cases, width=0.4, label='Cases', color='#CD5C5C', edgecolor='black')

# Customize
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_xlabel('Year')
ax.set_ylabel('Number of people')
ax.set_yscale('log')
ax.legend()

ax.annotate('West African epidemic',
            xy=(x[16], cases[16]/2),
            xytext=(x[16]-4, cases[16]/1.5),
            arrowprops=dict(facecolor='#1B4F72', shrink=0.05, width=1, headwidth=8),
            fontsize=16,
            ha='center')

plt.tight_layout()
save_figure('ebola_chart')

In [5]:
# Solution domain
dims = ((0, 1), (0, 1), (0, 1))

In [6]:
def domain(MK, dims=dims):
    M, K = MK
    h, ht = 1/M, 1/K

    z = np.linspace(dims[0][0], dims[0][1], M+1)
    r = np.linspace(dims[1][0], dims[1][1], M+1)
    t = np.linspace(dims[2][0], dims[2][1], K+1)

    ZZ, RR = np.meshgrid(z, r) 

    return (h, ht), (z, r, t), (ZZ, RR)

In [7]:
# Scaled block matrix coefficients
def matrix_coeffs(model, params):
    ht, h, r = model
    R, L, E, U, D = params

    δ = ht/h**2 * D
    γ = δ * (R/L)**2
    ξ = ht/h * D*(E-1)/r
    𝜗 = ht/h * U*(1-r**2)
    ω = ξ + 𝜗 + 2*(γ + δ) + 1

    return δ, γ, ξ, 𝜗, ω

In [8]:
def coeff_matrix(M:int, coeffs) -> sp.csr_matrix:
    '''
    Create the block tridiagonal matrix A to solve AC^k+1 + g = -C^k.
    Input:
        M: number of grid points
    '''
    δ, γ, ξ, 𝜗, ω = coeffs
    Z = M*(M-1)

    ll = np.repeat(δ + ξ[1:-1], M) # Lowest diag
    l = np.repeat(γ + 𝜗[1:-1], M)  # Next lowest diag
    d = np.repeat(ω[1:-1], M)      # Main diag 
    u = np.full(Z, γ)              # Next highest diag
    uu = np.full(Z, δ)             # Highest diag
    
    l[M-1::M], u[M-1::M] = 0, 0 # Every M'th element is zero in l and d
    
    A = sp.diags([ll, l, -d, u, uu], [-M, -1, 0, 1, M], (Z, Z), format='csr')

    return A

In [9]:
def boundary_vector(M, model:tuple, params, coeffs, k:int, qk:np.ndarray, Ck:np.ndarray, BC:tuple) -> np.ndarray:
  ''' 
  Create the boundary vector g to solve AC^k+1 + g = -C^k.
  Input:
    grid: (r, z) grids in r- and z directions
       k: the previous iteration step
      Ck: the previous concentration matrix
      BC: (g1, g2, g3) boundary condition functions
       M: number of grid points
  '''
  h, z, r, t = model
  δ, γ, ξ, 𝜗, ω = coeffs
  g1, g2, g3 = BC
  
  # Construct a matrix containing boundary contributions.
  G = np.zeros((M-1, M))
  G[::-1, 0] += (γ+𝜗[1:-1]) * g1(r[1:-1], t[k+1])       # Left contribution
  G[-1, :] += (δ+ξ[1]) * g2(z[1:], t[k+1])              # Bottom contribution
  G[0, :] += δ * g3((h, r), params, qk[1:], Ck[-2, 1:]) # Top contribution

  return G[::-1].ravel() # return G as a vector

In [10]:
def concentration_scheme(M, model:tuple, params, A:np.ndarray, g:np.ndarray, k:int, qk:np.ndarray, Ck:np.ndarray, BC:tuple) -> np.ndarray:
    '''
    Calculate the next concentration matrix C^k+1.
    Input:
        grid: (r, z) grids in r- and z directions
           A: the coefficient matrix
           k: the previous iteration step
          Ck: the previous concentration matrix
           g: the boundary vector
          BC: (g1, g2, g3) boundary condition functions
           M: number of grid points
    '''
    h, z, r, t = model
    g1, g2, g3 = BC
   
    C = np.zeros((M+1, M+1))
    C[::-1, 0] = g1(r, t[k+1]) # Left boundary
    C[-1, :] = g2(z, t[k+1])   # Bottom boundary  
    C[0, 1:] = g3((h, r), params, qk[1:], Ck[-2, 1:]) # Top boundary

    C_interior = sp.linalg.spsolve(A, -(g + Ck[1:-1, 1:][::-1].ravel()))
    C[1:-1, 1:] = np.reshape(C_interior, (M-1, M))
   
    return C

In [11]:
def RK4(f, q, C, params):
    ''' 
    Solve a set of ODE's using the RK4-method.
    Input:
        f: the right hand side of the differential equations. Here: The Langmuir model
        q: initial values
        C: concentration matrix
        params: parameters needed by f.
    '''
    ht, kA = params
    k1 = f(q, C, kA)
    k2 = f(q+ht*k1/2, C, kA)
    k3 = f(q+ht*k2/2, C, kA)
    k4 = f(q+ht*k3, C, kA)
    
    return q + ht/6*(k1 + 2*(k2 + k3) + k4)

In [12]:
# Initial condition
def f(z, r):                   
    return np.zeros_like(z)   
                                                         
# Boundary conditions
def i(r, t): # inlet
    return 1-r**2  

def e(z, t): # electric
    return 0

def langmuir(q, C, kA): # RHS of langmuir model to be solved with RK4
    return kA*C*(1-q)

def d(model, params, qk, C): # deposition (discretized)
    h, r = model
    R, E, D, kA = params
    return C / (1 - h*((E*R)/r[-1] + kA/D *(1-qk)))

### Simulator functions

In [13]:
def contour_plot(Z, R, C, dCz=None, dCr=None, skip=12, filename=None):
    # Plot base concentration contour
    contour = plt.contourf(Z, R, C, levels=200, cmap=concentration_cmap)
    cbar = plt.colorbar(contour, label=r'$c/c_{\mathrm{max}}$')

    if dCz is not None and dCr is not None:
        Z_skip   = Z[::skip, ::skip]   # Subsample for clarity
        R_skip   = R[::skip, ::skip]
        dCz_skip = dCz[::skip, ::skip]
        dCr_skip = dCr[::skip, ::skip]

        mag = np.sqrt(dCz_skip**2 + dCr_skip**2)

        # Enhance contrast in lengths using exponent
        exponent = 0.04
        mag_scaled = mag**exponent
        mag_scaled /= np.max(mag_scaled)  # Normalize to [0, 1]

        dCz_scaled = dCz_skip * mag_scaled / mag  # Scale vector components by adjusted magnitude
        dCr_scaled = dCr_skip * mag_scaled / mag

        arrow_length = 0.1      # Rescale to desired max arrow length
        dCz_final = dCz_scaled * arrow_length
        dCr_final = dCr_scaled * arrow_length

        plt.quiver(Z_skip, R_skip, dCz_final, dCr_final, width=0.003, color='slategrey', scale=1.5, scale_units='xy')

    plt.xlabel('$z/L$')
    plt.ylabel('$r/R$')
    plt.gca().set_aspect(aspect=0.6, adjustable='box')

    if filename is not None:
        save_figure(filename)
    else:
        save_figure('contour_plot')


In [14]:
def simulate_concentration(MK, model, params, label=None, show_contour=False, snapshot_times=(0.045, 0.145, 0.495), filename=None):
    M, K = MK
    (h, ht), (z, r, t), (ZZ, RR) = model
    R, L, E, U, D, kA = params

    coeffs = matrix_coeffs((ht, h, r), (R, L, E, U, D))
    A = coeff_matrix(M, coeffs)
    BC = (i, e, d)

    Ck = f(ZZ, RR)             # Initial concentration matrix
    qk = np.zeros(M+1)         # Initial wall surface coverage
    Cwall = np.zeros((K, M+1)) # Store wall consentrations for each step

    # Convert fractions of time domain into indices
    snapshot_indices = set(np.round(np.array(snapshot_times) * (K - 1)).astype(int))
    
    for k, time in enumerate(t[:-1]):
        g = boundary_vector(M, (h, z, r, t), (R, E, D, kA), coeffs, k, qk, Ck, BC)
        C = concentration_scheme(M, (h, z, r, t), (R, E, D, kA), A, g, k, qk, Ck, BC)
        dCr, dCz = np.gradient(C, h, h)

        Cwall[k] = C[0]
        q = RK4(langmuir, qk, C[-1, :], (ht, kA))

        if show_contour and k in snapshot_indices:
            plt.figure()
            plt.title(f'$t/T = {time+ht:.2f}$')
            if label:
                filename = f'{label}_t{time+ht:.2f}'
            else:
                filename = f'contour_t{time+ht:.2f}'
            contour_plot(ZZ, RR, C, -dCz, -dCr, filename=filename) # Note: negative gradient is flow direction

        Ck = C
        qk = q

    return Cwall

In [15]:
def simulate_deposition(K, model, Cwall, profiles=20, filename=None):
    z, t = model

    plt.figure()
    # Select "profiles" evenly spaced indices from the time steps
    profile_indices = np.round(np.linspace(0, K-1, profiles)).astype(int)

    for j, k in enumerate(profile_indices):
        color = deposition_cmap(j/max(1, profiles-1))  # Normalize color mapping
        plt.plot(z, Cwall[k], color=color)

    plt.xlabel('$z/L$')
    plt.ylabel(r'$c/c_{\mathrm{max}}$')
    plt.grid(True)

    sm = plt.cm.ScalarMappable(cmap=deposition_cmap, norm=plt.Normalize(vmin=t[0], vmax=t[-1]))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=plt.gca())
    cbar.set_label('$t/T$')

    if filename is not None:
        save_figure(filename)
    else:
        save_figure('deposition_profile')

### Introductory examples

In [16]:
M, K = 200, 200
(h, ht), (z, r, t), (ZZ, RR) = domain((M, K))

In [ ]:
# Example 1: Varying E, contour plots, deposition profile
R, L, U, D, kA = 0.1, 1, 6, 1, 3
E_list = [1, 2, 3]
for E in E_list:
    Cwall = simulate_concentration((M, K), ((h, ht), (z, r, t), (ZZ, RR)), (R, L, E, U, D, kA), label=f'E{E:.1f}', show_contour=True)
    simulate_deposition(K, (z, t), Cwall, filename=f'deposition_E{E:.1f}')

In [18]:
# Example 2: Varying U, contour plots, deposition profile
R, L, E, D, kA = 0.1, 1, 2, 1, 3
U_list = [1, 5, 10]
for U in U_list:
    Cwall = simulate_concentration((M, K), ((h, ht), (z, r, t), (ZZ, RR)), (R, L, E, U, D, kA), label=f'U{U:.1f}', show_contour=True)
    simulate_deposition(K, (z, t), Cwall, filename=f'deposition_U{U:.1f}')

In [19]:
# Example 3: Varying D, contourplots, deposition profile
R, L, E, U, kA = 0.1, 1, 2, 6, 3
D_list = [0.5, 2, 3]
for D in D_list:
    Cwall = simulate_concentration((M, K), ((h, ht), (z, r, t), (ZZ, RR)), (R, L, E, U, D, kA), label=f'D{D:.1f}', show_contour=True)
    simulate_deposition(K, (z, t), Cwall, filename=f'deposition_D{D:.1f}')

In [20]:
# Example 4: Varying kA, contour plots, deposition profile
R, L, E, U, D = 0.1, 1, 2, 6, 1
kA_list = [0.5, 2, 5]
for kA in kA_list:
    Cwall = simulate_concentration((M, K), ((h, ht), (z, r, t), (ZZ, RR)), (R, L, E, U, D, kA), label=f'kA{kA:.1f}', show_contour=True)
    simulate_deposition(K, (z, t), Cwall, filename=f'deposition_kA{kA:.1f}')

In [21]:
# Example 5: Holding tube radius constant, varying length
R, E, U, D, kA = 0.1, 2, 6, 1, 3
L_list = [2*R, 5*R, 15*R, 25*R]
for L in L_list:
    Cwall = simulate_concentration((M, K), ((h, ht), (z, r, t), (ZZ, RR)), (R, L, E, U, D, kA))
    simulate_deposition(K, (z, t), Cwall, filename=f'deposition_L{L:.1f}')

### Manufactured solution

In [17]:
def S(M, r, z, t, params):
    R, L, E, U, D = params
    
    term_time = -np.sin(np.pi*r) * np.cos(np.pi*z)
    term_diff = D * np.pi**2 * np.sin(np.pi*r) * np.cos(np.pi*z) * (1 + (R/L)**2)
    term_radial = D * (E - 1) * (1/r) * np.pi * np.cos(np.pi*r) * np.cos(np.pi*z)
    term_conv = U * (1 - r**2) * (-np.pi * np.sin(np.pi*z) * np.sin(np.pi*r))

    return (term_time + term_diff + term_radial + term_conv) * np.exp(-t)

In [18]:
def coeff_matrix_ms(M:int, coeffs) -> sp.csr_matrix:
    δ, γ, ξ, 𝜗, ω = coeffs
    Z = (M-1)**2

    ll = np.repeat(δ + ξ[1:-1], M-1) # Lowest diag
    l = np.repeat(γ + 𝜗[1:-1], M-1)  # Next lowest diag
    d = np.repeat(ω[1:-1], M-1)      # Main diag 
    u = np.full(Z, γ)                # Next highest diag
    uu = np.full(Z, δ)               # Highest diag
    
    l[M-2::M-1], u[M-2::M-1] = 0, 0 # Every M-1'th element is zero in l and d
    
    A = sp.diags([ll, l, -d, u, uu], [-(M-1), -1, 0, 1, (M-1)], (Z, Z), format='csr')

    return A

In [19]:
def boundary_vector_ms(M, model:tuple, coeffs, k:int, Ck:np.ndarray, BC:tuple) -> np.ndarray:
  z, r, t = model
  δ, γ, ξ, 𝜗, ω = coeffs
  g1, g2, g3, g4 = BC
  
  # Construct a matrix containing boundary contributions.
  G = np.zeros((M-1, M-1))
  G[::-1, 0] += (γ+𝜗[1:-1]) * g1(r[1:-1], t[k+1]) # Left contribution
  G[-1, :] += (δ+ξ[1]) * g2(z[1:-1], t[k+1])      # Bottom contribution
  G[0, :] += δ * g3(z[1:-1], t[k+1])              # Top contribution
  G[::-1, -1] += γ * g4(r[1:-1], t[k+1])          # Right contribution

  return G[::-1].ravel() # return G as a vector

In [20]:
def concentration_scheme_ms(M, model, params, A:np.ndarray, g:np.ndarray, k:int, Ck:np.ndarray, BC:tuple, S) -> np.ndarray:
    (h, ht), (z, r, t), (ZZ, RR) = model
    g1, g2, g3, g4 = BC
   
    C = np.zeros((M+1, M+1))
    C[::-1, 0] = g1(r, t[k+1])     # Left boundary
    C[-1, :] = g2(z, t[k+1])       # Bottom boundary  
    C[0, 1:] = g3(z[1:-1], t[k+1]) # Top boundary
    C[::-1, -1] = g4(r, t[k+1])    # Right boundary
    
    C_interior = sp.linalg.spsolve(A, -(g + Ck[1:-1, 1:-1][::-1].ravel() + ht*S(M, RR, ZZ, t[k+1], params)[1:-1, 1:-1][::-1].ravel()))
    C[1:-1, 1:-1] = np.reshape(C_interior, (M-1, M-1))
   
    return C

In [21]:
# Initial condition
def f_ms(r, z):                   
    return np.sin(np.pi*r)*np.cos(np.pi*z)   
                                                         
# Boundary conditions
def i_ms(r, t): # inlet
    return np.sin(np.pi*r)*np.exp(-t)

def e_ms(z, t): # electric
    return 0

def d_ms(z, t): # top
    return 0

def o_ms(r, t): # outlet
    return -np.sin(np.pi*r)*np.exp(-t)

def c_exact(r, z, t):
    return np.sin(np.pi*r)*np.cos(np.pi*z)*np.exp(-t)

### Convergence tests

In [22]:
def spatial_converge(params, M_list, K):
    R, L, E, U, D = params
    BC = (i_ms, e_ms, d_ms, o_ms)

    errors_Linf , errors_L2 = np.zeros(len(M_list)), np.zeros(len(M_list))
    H = np.zeros(len(M_list))
    
    for j, M in enumerate(M_list):
        (h, ht), (z, r, t), (ZZ, RR) = domain((M, K))

        coeffs = matrix_coeffs((ht, h, r), (R, L, E, U, D))
        A = coeff_matrix_ms(M, coeffs)

        Ck = f_ms(RR, ZZ)
        for k in range(K-1):
            g = boundary_vector_ms(M, (z, r, t), coeffs, k, Ck, BC)
            Ck = concentration_scheme_ms(M, ((h, ht), (z, r, t), (ZZ, RR)), (R, L, E, U, D), A, g, k, Ck, BC, S)

        C_exact = c_exact(RR, ZZ, 1)
        error = C_exact - Ck

        errors_Linf[j] = np.linalg.norm(error, ord=np.inf)
        errors_L2[j] = h * np.linalg.norm(error, ord='fro')
        H[j] = h

    p_Linf = np.polyfit(np.log(H), np.log(errors_Linf), 1)[0]
    p_L2 = np.polyfit(np.log(H), np.log(errors_L2), 1)[0]

    return H, errors_Linf, errors_L2, p_Linf, p_L2

In [23]:
def temporal_converge(params, K_list, M):
    R, L, E, U, D = params
    BC = (i_ms, e_ms, d_ms, o_ms)

    errors_Linf , errors_L2 = np.zeros(len(K_list)), np.zeros(len(K_list))
    Ht = np.zeros(len(K_list))

    for j, K in enumerate(K_list):
        (h, ht), (z, r, t), (ZZ, RR) = domain((M, K))

        coeffs = matrix_coeffs((ht, h, r), (R, L, E, U, D))
        A = coeff_matrix_ms(M, coeffs)

        Ck = f_ms(RR, ZZ)
        for k in range(K-1):
            g = boundary_vector_ms(M, (z, r, t), coeffs, k, Ck, BC)
            Ck = concentration_scheme_ms(M, ((h, ht), (z, r, t), (ZZ, RR)), (R, L, E, U, D), A, g, k, Ck, BC, S)

        C_exact = c_exact(RR, ZZ, 1)
        error = C_exact - Ck

        errors_Linf[j] = np.linalg.norm(error, ord=np.inf)
        errors_L2[j] = h * np.linalg.norm(error, ord='fro')
        Ht[j] = ht
        
    p_Linf = np.polyfit(np.log(Ht), np.log(errors_Linf), 1)[0]
    p_L2 = np.polyfit(np.log(Ht), np.log(errors_L2), 1)[0]

    return Ht, errors_Linf, errors_L2, p_Linf, p_L2

In [24]:
def plot_convergence(converge_func, params, resolution_list, fixed_resolution, filename=None):
    # Call the appropriate convergence function
    H, errors_Linf, errors_L2, p_Linf, p_L2 = converge_func(params, resolution_list, fixed_resolution)

    # Determine label base depending on which convergence function is being used
    label_base = "h" if converge_func.__name__ == 'spatial_converge' else 'h_t'

    plt.loglog(H, errors_Linf, 'o-', c='#1B4F72', label=fr'$p^{{L^{{\infty}}}}_{{{label_base}}} = {p_Linf:.2f}$')
    plt.loglog(H, errors_L2, 'o-', c='#8B3A3A', label=fr'$p^{{L^2}}_{{{label_base}}} = {p_L2:.2f}$')
    plt.xlabel(f'${label_base}$')
    plt.ylabel(f'$||e({label_base})||$')
    plt.legend()
    plt.grid(True)
    
    if filename is not None:
        save_figure(filename)
    else:
        save_figure('convergence_plot')

In [25]:
# Example 1: Naive parameters
R, L, E, U, D = 0.1, 1, 3, 4, 1
plot_convergence(spatial_converge, (R, L, E, U, D), [8, 16, 32, 64], 5000, filename='spat_naive')

In [26]:
R, L, E, U, D = 0.1, 1, 3, 4, 1
plot_convergence(temporal_converge, (R, L, E, U, D), [25, 50, 100, 200], 70, filename='temp_naive')

In [27]:
# Example 2: Optimized parameters
R, L, E, U, D = 9, 2, 9, 20, 0.00125
plot_convergence(spatial_converge, (R, L, E, U, D), [8, 16, 32, 64], 5000, filename='spat_opt')

In [28]:
R, L, E, U, D = 3, 2, 9, 20, 0.0125
plot_convergence(temporal_converge, (R, L, E, U, D), [25, 50, 100, 200], 70, filename='temp_opt')

### The Pareto Front

In [77]:
def read_convergence_data(filename):
    '''
    Read convergence data from filename directly into numpy arrays.
    '''
    with open(filename, 'r') as f:
        lines = f.readlines()

    n_blocks = len(lines) // 3  # Every 3 lines is a block

    # Preallocate arrays
    E_array = np.zeros(n_blocks)
    U_array = np.zeros(n_blocks)
    D_array = np.zeros(n_blocks)
    kA_array = np.zeros(n_blocks)
    p_temp_array = np.zeros((n_blocks, 2))
    p_spat_array = np.zeros((n_blocks, 2))

    for idx in range(n_blocks):
        i = idx * 3

        # Parse E, U, D, kA
        EUDkA_line = lines[i]
        parts = EUDkA_line.split(',')
        E_array[idx] = float(parts[0].split(':')[1])
        U_array[idx] = float(parts[1].split(':')[1])
        D_array[idx] = float(parts[2].split(':')[1])
        kA_array[idx] = float(parts[3].split(':')[1])

        # Parse Temp converge output
        temp_line = lines[i+1]
        temp_parts = temp_line.split('np.float64')
        p_temp_array[idx, 0] = float(temp_parts[1].replace(')', '').replace('(', '').replace(',', '').strip())
        p_temp_array[idx, 1] = float(temp_parts[2].replace(')', '').replace('(', '').replace(',', '').strip())

        # Parse Spat converge output
        spat_line = lines[i+2]
        spat_parts = spat_line.split('np.float64')
        p_spat_array[idx, 0] = float(spat_parts[1].replace(')', '').replace('(', '').replace(',', '').strip())
        p_spat_array[idx, 1] = float(spat_parts[2].replace(')', '').replace('(', '').replace(',', '').strip())

    return E_array, U_array, D_array, kA_array, p_temp_array, p_spat_array

In [78]:
E_array, U_array, D_array, kA_array, p_temp, p_spat = read_convergence_data('C:\\Users\\mikai\\OneDrive\\Skrivebord\\fysmat\\TMA4850\\Python\\opt_params.txt')

In [57]:
def plot_params(E, U, D, kA, filename=None):
    import matplotlib.pyplot as plt
    import numpy as np
    import matplotlib.colors as mcolors

    n_sets = len(E)
    x_positions = np.array([0, 1, 2, 3])  # E, U, D, kA
    labels = ['$E$', '$U$', '$D$', '$k_a$']

    colors = deposition_cmap(np.linspace(0, 1, n_sets))

    fig, ax = plt.subplots()
    for i in range(n_sets):
        y_values = [E[i], U[i], D[i], kA[i]]

        # Jitter x slightly to prevent overlap
        jitter = np.random.normal(0, 0.15, size=4)  # 4 values for E, U, D, kA
        jittered_x = x_positions + jitter

        ax.scatter(jittered_x, y_values, color=colors[i], s=100, alpha=0.8, edgecolor='k', linewidth=0.5)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Magnitude')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    if filename is not None:
        save_figure(filename)
    else:
        save_figure('param_plot')

In [79]:
plot_params(E_array, U_array, D_array, kA_array, filename='pareto_params')
print(E_array[-1], U_array[-1], D_array[-1], kA_array[-1])

2.8614588969002974 10.089353537323369 0.9939968523309523 5.776605886455192


### Optimal examples

In [ ]:
M, K = 200, 200
(h, ht), (z, r, t), (ZZ, RR) = domain((M, K))
R, L, E, U, D, kA = 0.1, 1, E_array[-1], U_array[-1], D_array[-1], kA_array[-1]

# Example: Deposition profile of the means
Cwall = simulate_concentration((M, K), ((h, ht), (z, r, t), (ZZ, RR)), (R, L, E, U, D, kA))
simulate_deposition(K, (z, t), Cwall, filename=f'deposition_pareto')